In [17]:
from sklearn.metrics import classification_report

"""
案例:
    电信客户流失分析.

目的:
    1. 演示逻辑回归的相关操作, 主要是: 二分法(流失, 不流失)
    2. 演示逻辑回归的评估操作, 主要是: 混淆矩阵, 准确率, 召回率, F1值, ROC曲线, AUC值, 分类评估报告(了解)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score


# 1. 定义函数, 用于实现数据预处理
def demo01_data_preprocess():
    # 1. 读取CSV文件, 获取df对象
    churn_df = pd.read_csv('../data/churn.csv')
    # 2. 查看(处理前)的数据
    #churn_df.info()
    #print(churn_df.head())
    # 3. 因为Churn 和 gender列是自渡川, 所以需要进行one-hot编码(热编码处理)
    churn_df = pd.get_dummies(churn_df, columns=['Churn', 'gender'])
    # 4. 查看(处理后)的数据集
    #print(churn_df.info())
    # 5. 删除one-hot处理后, 冗余的列
    # 参1: 要删除的列  参2: 1表示删除列  参3: 是否直接修改原数据
    churn_df.drop(['Churn_No', 'gender_Male'], axis=1, inplace=True)
    #print(churn_df.head())
    # 6. 修改列名,将Churn_Yes修改为flag
    churn_df.rename(columns={'Churn_Yes': 'flag'}, inplace=True)
    print(churn_df.head())  # False --> 不流失,  True --> 流失

    # 7. 查看数据值的分布
    print(churn_df.flag.value_counts())  # False: 5174 , True: 1869


# 2. 定义函数, 演示: 数据的可视化
def demo02_data_visualization():
    # 1~6: 数据预处理
    churn_df = pd.read_csv('../data/churn.csv')
    churn_df = pd.get_dummies(churn_df, columns=['Churn', 'gender'])
    churn_df.drop(['Churn_No', 'gender_Male'], axis=1, inplace=True)
    churn_df.rename(columns={'Churn_Yes': 'flag'}, inplace=True)
    # 7. 数据的可视化, 绘制 计数柱状图
    # 参1: 数据集  参2:x轴的列名(月度会员)  参3:hue: 表示分组,根据分组进行绘制
    sns.countplot(data=churn_df, x='Contract_Month', hue='flag')
    plt.show()


# 3. 模型评测和训练
def demo03_model_evaluation_and_training():
    # 1: 数据预处理
    churn_df = pd.read_csv('../data/churn.csv')
    churn_df = pd.get_dummies(churn_df, columns=['Churn', 'gender'])
    churn_df.drop(['Churn_No', 'gender_Male'], axis=1, inplace=True)
    churn_df.rename(columns={'Churn_Yes': 'flag'}, inplace=True)

    # 2.1 提取特征列和标签列
    # x的特征列: 月度会员,是否有互联网服务, 是否电子支付
    x = churn_df[['Contract_Month', 'internet_other', 'PaymentElectronic']]
    y = churn_df['flag']
    # 2.2 划分训练集和测试集
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=20)

    # 3. 特征工程(暂不处理)

    # 4. 模型训练
    # 4.1 创建(逻辑回归)模型对象
    estimator  = LogisticRegression()
    # 4.2 模型训练
    estimator.fit(x_train, y_train)

    # 5. 模型预测
    y_pred = estimator.predict(x_test)
    print(f'模型预测值: {y_pred[:5]}')

    # 6. 模型评估
    print(f'准确率: {estimator.score(x_test, y_test)}')    # 预测前
    print(f'准确率: {accuracy_score(y_test, y_pred)}')     # 预测后

    print(f"精确率: {precision_score(y_test, y_pred)}")
    print(f"召回率: {recall_score(y_test, y_pred)}")
    print(f"F1值: {f1_score(y_test, y_pred)}")

    # macro avg : 表示宏平均值, 即: 不考虑样本权重, 对每个类别的指标取平均值, 适用于: 数据均很的情况.
    # weighted avg : 表示加权平均值, 即: 考虑样本权重, 对每个类别的指标取平均值, 适用于: 数据不均衡的情况.
    print(f'分类评估报告: \n{classification_report(y_test, y_pred)}')



if __name__ == '__main__':
    # demo01_data_preprocess()
    # demo02_data_visualization()
    demo03_model_evaluation_and_training()

模型预测值: [False False False False  True]
准确率: 0.8034066713981547
准确率: 0.8034066713981547
精确率: 0.632183908045977
召回率: 0.476878612716763
F1值: 0.5436573311367381
分类评估报告: 
              precision    recall  f1-score   support

       False       0.84      0.91      0.87      1063
        True       0.63      0.48      0.54       346

    accuracy                           0.80      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.79      0.80      0.79      1409

